
..# Part A Density-First Model

Patch density training with preprocessing comparison and validation-based strategy selection.


In [1]:
class Cfg:
 gz:tuple[int,int]=(896,896); ds:int=4; bs:int=4; ep:int=28; lr:float=2.0e-4; wd:float=1e-4; vf:float=0.15; modes:tuple[str,...]=('fused',); seed:int=42
 bounds:tuple[float,float]=(150.0,520.0); dense_cut:float=620.0; overlap:float=160.0; sparse_cap:float=320.0; low_cap:float=720.0; dense_cap:float=2600.0
 sparse_cut:float=220.0; sparse_use_max:float=280.0; sparse_guard:float=320.0; sparse_strict:float=250.0; dense_trigger:float=0.82; dense_margin:float=0.04; dense_force:float=0.93
cfg=Cfg(); cfg


In [2]:
import math, random
import cv2, h5py, numpy as np, pandas as pd, torch
from copy import deepcopy
from pathlib import Path
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
R=globals().get('R',Path.cwd())
if not (R/'DATASET').exists() and (R.parent/'DATASET').exists(): R=R.parent
D=globals().get('D',R/'DATASET')
M=globals().get('M',R/'processed_density_maps')
if not M.exists() and (R/'ground_truth').exists(): M=R/'ground_truth'
OUT=globals().get('OUT',R/'prediction_results_part_a.csv')
DEV=globals().get('DEV',torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
def seed(x):
 random.seed(x); np.random.seed(x); torch.manual_seed(x)
 if torch.cuda.is_available(): torch.cuda.manual_seed_all(x)
def samples(part='A',split='train'):
 xs=[]; idir=D/f'part_{part}'/('train_data' if split=='train' else 'test_data')/'images'; ddir=M/f'part_{part}'/split
 for ip in sorted(idir.glob('IMG_*.jpg')):
  hp=ddir/f'{ip.stem}.h5'
  if hp.exists(): xs.append({'split':split,'image_id':ip.stem,'image_path':ip,'density_path':hp})
 return xs
img=lambda p: np.asarray(Image.open(p).convert('RGB'),dtype=np.uint8)
def den(p):
 with h5py.File(p,'r') as h: return h['density'][:].astype(np.float32)
cnt=lambda s: float(den(s['density_path']).sum())
def split(xs,vf=0.15,seedv=42):
 xs=list(xs); cs=[cnt(x) for x in xs]; xs=[x for _,x in sorted(zip(cs,xs),key=lambda t:t[0])]; step=max(2,round(1/max(vf,1e-6))); off=seedv%step
 va=[x for i,x in enumerate(xs) if i%step==off]; tr=[x for i,x in enumerate(xs) if i%step!=off]
 return tr,va
def bucket(c,bounds=None):
 b=cfg.bounds if bounds is None else bounds
 if c<b[0]: return 0
 if c<b[1]: return 1
 return 2
def rsz(d,w,h):
 z=cv2.resize(d,(w,h),interpolation=cv2.INTER_CUBIC).astype(np.float32); a=float(d.sum()); b=float(z.sum())
 if a>0 and b>0: z*=a/b
 return z
def prep(x,mode):
 rgb=x.astype(np.float32)/255.0
 g=cv2.cvtColor(x,cv2.COLOR_RGB2GRAY); c=np.repeat(cv2.createCLAHE(2.0,(8,8)).apply(g)[...,None],3,2).astype(np.float32)/255.0
 if mode=='rgb': return rgb
 if mode=='gray_clahe': return c
 return np.concatenate([rgb,c],2)
def dense_lbl(c): return int(float(c)>=cfg.dense_cut)
def filter_split(xs,kind):
 if kind=='sparse': return [x for x in xs if cnt(x)<=cfg.sparse_cut+60.0]
 if kind=='low': return [x for x in xs if cnt(x)<=cfg.dense_cut+cfg.overlap]
 if kind=='dense': return [x for x in xs if cnt(x)>=cfg.dense_cut-cfg.overlap]
 return list(xs)
class FullDS(Dataset):
 def __init__(self,xs,mode,z=(896,896),ds=4,aug=False):
  self.xs=list(xs); self.mode=mode; self.z=z; self.ds=ds; self.aug=aug; self.q=(z[0]//ds,z[1]//ds)
 def __len__(self): return len(self.xs)
 def __getitem__(self,i):
  s=self.xs[i]; x=img(s['image_path']); d=den(s['density_path']); c=float(d.sum()); b=bucket(c); rb=dense_lbl(c)
  if self.aug and random.random()<0.5: x=np.ascontiguousarray(np.fliplr(x)); d=np.ascontiguousarray(np.fliplr(d))
  if self.aug and random.random()<0.2: x=np.ascontiguousarray(np.flipud(x)); d=np.ascontiguousarray(np.flipud(d))
  if self.aug and random.random()<0.5: x=np.clip(x.astype(np.float32)*random.uniform(.9,1.1)+random.uniform(-12,12),0,255).astype(np.uint8)
  h,w=x.shape[:2]
  if self.aug and min(h,w)>320 and random.random()<0.45:
   ch=random.randint(int(.72*h),h); cw=random.randint(int(.72*w),w); t=random.randint(0,h-ch); l=random.randint(0,w-cw); x=x[t:t+ch,l:l+cw]; d=d[t:t+ch,l:l+cw]
  x=cv2.resize(x,self.z,interpolation=cv2.INTER_AREA); d=rsz(d,self.q[1],self.q[0])
  return torch.from_numpy(prep(x,self.mode).transpose(2,0,1)), torch.from_numpy(d).unsqueeze(0), torch.tensor([math.log1p(c)],dtype=torch.float32), torch.tensor([math.sqrt(max(c,0.0))],dtype=torch.float32), torch.tensor([c],dtype=torch.float32), torch.tensor(b,dtype=torch.long), torch.tensor(rb,dtype=torch.float32)
class SE(nn.Module):
 def __init__(self,c,r=8):
  super().__init__(); h=max(8,c//r); self.m=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(c,h,1),nn.ReLU(True),nn.Conv2d(h,c,1),nn.Sigmoid())
 def forward(self,x): return x*self.m(x)
class RB(nn.Module):
 def __init__(self,a,b,s=1):
  super().__init__(); self.p=nn.Identity() if (a==b and s==1) else nn.Sequential(nn.Conv2d(a,b,1,s,0),nn.BatchNorm2d(b))
  self.m=nn.Sequential(nn.Conv2d(a,b,3,s,1),nn.BatchNorm2d(b),nn.ReLU(True),nn.Conv2d(b,b,3,1,1),nn.BatchNorm2d(b),SE(b)); self.r=nn.ReLU(True)
 def forward(self,x): return self.r(self.p(x)+self.m(x))
class CountNet(nn.Module):
 def __init__(self,out='log'):
  super().__init__(); self.out=out; ch=6 if 'fused' in cfg.modes else 3
  self.enc=nn.Sequential(RB(ch,32,2),RB(32,64,2),RB(64,96,1),RB(96,128,2),RB(128,160,1),RB(160,192,1))
  self.den=nn.Sequential(nn.Conv2d(192,128,3,1,1),nn.ReLU(True),nn.Conv2d(128,64,3,1,1),nn.ReLU(True),nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),nn.Conv2d(64,1,1),nn.Softplus())
  self.pool=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten())
  self.head=nn.Sequential(nn.Linear(192,128),nn.ReLU(True),nn.Dropout(.2),nn.Linear(128,1))
  self.bh=nn.Sequential(nn.Linear(192,64),nn.ReLU(True),nn.Dropout(.1),nn.Linear(64,3))
 def forward(self,x,aux=False):
  f=self.enc(x); d=self.den(f); g=self.pool(f); z=self.head(g); logits=self.bh(g)
  if aux: return d,z,logits
  return z
class RouterNet(nn.Module):
 def __init__(self):
  super().__init__(); ch=6 if 'fused' in cfg.modes else 3
  self.m=nn.Sequential(RB(ch,24,2),RB(24,48,2),RB(48,72,2),RB(72,96,2),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(96,64),nn.ReLU(True),nn.Dropout(.1),nn.Linear(64,1))
 def forward(self,x): return self.m(x).reshape(-1)
def inv_count(z,kind):
 if kind=='sqrt': return torch.clamp(z.reshape(-1),min=0)**2
 return torch.expm1(z.reshape(-1))
def tgt_count(logc,sqrtc,kind):
 return sqrtc.reshape(-1) if kind=='sqrt' else logc.reshape(-1)
def sample_weight(c,kind):
 c=np.asarray(c,np.float32)
 if kind=='sparse': return np.clip(1.0+0.0010*np.minimum(c,cfg.sparse_cut+80.0),1.0,1.7)
 if kind=='dense': return np.clip(1.2+0.0022*c+0.40*(c>900)+0.55*(c>1400),1.2,6.5)
 return np.clip(1.0+0.0012*np.minimum(c,cfg.dense_cut+200.0),1.0,2.2)
def ep_count(model,ld,opt=None,kind='log'):
 tr=opt is not None; model.train(tr); pix=nn.SmoothL1Loss(beta=.25); reg=nn.SmoothL1Loss(beta=.22,reduction='none'); clf=nn.CrossEntropyLoss(); tl=ta=trm=n=0
 for x,y,logc,sqrtc,c,b,_ in ld:
  x=x.to(DEV); y=y.to(DEV); logc=logc.to(DEV); sqrtc=sqrtc.to(DEV); c=c.to(DEV).reshape(-1); b=b.to(DEV)
  with torch.set_grad_enabled(tr):
   d,z,logits=model(x,True); dens=d.flatten(1).sum(1); pred=inv_count(z,kind); targ=tgt_count(logc,sqrtc,kind); dens_targ=torch.sqrt(torch.clamp(c,min=0)) if kind=='sqrt' else torch.log1p(torch.clamp(c,min=0))
   dens_pred=torch.sqrt(torch.clamp(dens,min=0)+1e-6) if kind=='sqrt' else torch.log1p(torch.clamp(dens,min=0))
   w=torch.tensor(sample_weight(c.detach().cpu().numpy(),kind),device=DEV,dtype=torch.float32)
   loss=.15*pix(d,y)+.58*(reg(z.reshape(-1),targ)*w).mean()+.17*((reg(pred,c)/(20.0+c))*w).mean()+.05*(reg(dens_pred,dens_targ)*torch.sqrt(w)).mean()+.05*clf(logits,b)
   if tr: opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
  e=pred.detach()-c; bs=x.size(0); tl+=float(loss.detach())*bs; ta+=float(e.abs().sum()); trm+=float((e**2).sum()); n+=bs
 return {'loss':tl/max(n,1),'mae':ta/max(n,1),'rmse':math.sqrt(trm/max(n,1))}
def ep_router(model,ld,opt=None):
 tr=opt is not None; model.train(tr); bce=nn.BCEWithLogitsLoss(reduction='none'); tl=ta=n=0
 for x,_,_,_,c,_,rb in ld:
  x=x.to(DEV); c=c.to(DEV).reshape(-1); rb=rb.to(DEV).reshape(-1)
  with torch.set_grad_enabled(tr):
   z=model(x)
   cc=c.detach().cpu().numpy()
   w=np.clip(1.0+0.002*np.abs(cc-cfg.dense_cut),1.0,3.0)
   w=w+1.2*((cc<cfg.bounds[1])&(rb.detach().cpu().numpy()<0.5))
   w=w+1.8*((cc<cfg.bounds[0])&(rb.detach().cpu().numpy()<0.5))
   w=torch.tensor(w,device=DEV,dtype=torch.float32)
   loss=(bce(z,rb)*w).mean()
   if tr: opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
  p=torch.sigmoid(z).detach(); e=(p>0.5).float()!=rb; bs=x.size(0); tl+=float(loss.detach())*bs; ta+=float(e.float().sum()); n+=bs
 return {'loss':tl/max(n,1),'err':ta/max(n,1)}
def infer_count(model,pth,mode,kind):
 x=img(pth); x=cv2.resize(x,cfg.gz,interpolation=cv2.INTER_AREA); u=torch.from_numpy(prep(x,mode).transpose(2,0,1)).unsqueeze(0).to(DEV)
 with torch.no_grad(): d,z,logits=model(u,True)
 dens=float(d.reshape(-1).sum().cpu().item()); raw=max(0.0,float(inv_count(z,kind).cpu().item())); p=torch.softmax(logits,1).cpu().numpy()[0].astype(np.float32)
 return {'raw':raw,'density':dens,'bucket':int(np.argmax(p))}
def infer_router(model,pth,mode):
 x=img(pth); x=cv2.resize(x,cfg.gz,interpolation=cv2.INTER_AREA); u=torch.from_numpy(prep(x,mode).transpose(2,0,1)).unsqueeze(0).to(DEV)
 with torch.no_grad(): z=model(u)
 return float(torch.sigmoid(z).cpu().item())
def fit_scale(a,p,mask,lo,hi):
 m=mask&(p>1.0)
 if int(m.sum())<4: return 1.0
 r=float(np.median(a[m]/np.clip(p[m],1.0,None)))
 return float(np.clip(r,lo,hi))
def route_clip(v,is_dense):
 if is_dense: return float(np.clip(v,cfg.bounds[1]*0.8,cfg.dense_cap))
 if v<cfg.bounds[0]: return float(np.clip(v,0.0,cfg.sparse_cap))
 return float(np.clip(v,0.0,cfg.low_cap))
def main_clip(v):
 return float(np.clip(v,0.0,cfg.dense_cap))
def score(a,p):
 e=np.abs(p-a); med=float(np.median(e)); trim=float(np.sort(e)[:max(1,int(math.ceil(.9*len(e))))].mean()); p90=float(np.quantile(e,.9)); bias=float((p-a).mean()); mae=float(e.mean())
 sparse=a<120; dense=a>900; so=float(np.maximum(p[sparse]-a[sparse],0).mean()) if sparse.any() else 0.0; du=float(np.maximum(a[dense]-p[dense],0).mean()) if dense.any() else 0.0
 s=.28*med+.18*trim+.12*p90+.12*abs(bias)+.14*so+.16*du
 return (s,med,p90,abs(bias)),{'val_score':s,'val_mae':mae,'val_medae':med,'val_p90ae':p90,'val_trimmed_mae':trim,'val_bias':bias,'val_sparse_over':so,'val_dense_under':du}
def merge(fs):
 z=fs[0]
 for f in fs[1:]: z=z.merge(f,on=['image_id','actual_count'],how='inner')
 return z
def smooth_calibrate(raw,scale,gamma,bias,cap):
 x=max(0.0,float(raw))
 return float(np.clip(scale*(x**gamma)+bias,0.0,cap))
def guarded_calibrate(sparse,low,dense,c):
 sparse=max(0.0,float(sparse)); low=max(0.0,float(low)); dense=max(0.0,float(dense))
 base=smooth_calibrate(low,c['scale'],c['gamma'],c['bias'],c['cap'])
 p=base
 low_ref=max(low,1.0); sparse_ref=max(sparse,1.0)
 dense_to_low=dense/low_ref
 dense_to_sparse=dense/sparse_ref
 aux_hallucination=(dense>c['halluc_aux'] and (low<c['halluc_low'] or sparse<c['halluc_sparse'] or dense_to_low>c['halluc_ratio'] or dense_to_sparse>c['halluc_sparse_ratio']))
 if low<c['base_low_cap_raw'] and sparse<c['base_sparse_cap_raw']:
  p=min(p,c['base_cap'])
 rescue_ok=(not aux_hallucination) and low>c['res_low'] and sparse>c['res_sparse'] and dense>c['res_aux'] and dense_to_low<c['res_ratio'] and dense_to_sparse<c['res_sparse_ratio']
 if rescue_ok:
  p=max(p,dense*c['res_mul'])
 if sparse<c['corr_sparse'] and low>c['corr_hi_low'] and dense>c['corr_dense'] and low>dense*c['corr_hi_ratio']:
  p=min(p,dense*c['corr_hi_mul'])
 if sparse<c['lift_sparse'] and low<c['lift_low'] and dense>c['lift_dense'] and dense_to_low>c['lift_ratio']:
  p=max(p,min(c['lift_cap'],dense*c['lift_mul']))
 if dense>c['extreme_dense'] and low>c['extreme_low']:
  p=max(p,min(c['extreme_cap'],dense*c['extreme_mul']))
 if p>c['hi_raw'] and dense<c['lo_aux']:
  p=min(p,dense*c['lo_mul'])
 if p>c['spike_raw'] and dense>c['spike_aux']:
  p=min(p,c['spike_cap'])
 if dense_to_low>c['hard_ratio'] and low<c['hard_low']:
  p=min(p,base)
 return float(np.clip(p,0.0,c['cap']))
def choose(v,modes):
 mode=modes[0]; a=v.actual_count.to_numpy(np.float32); sparse=v[f'sparse_{mode}'].to_numpy(np.float32); low=v[f'low_{mode}'].to_numpy(np.float32); dense=v[f'dense_{mode}'].to_numpy(np.float32); rows=[]; best=None
 candidates=(
  {'name':'strict','scale':0.96,'gamma':1.0,'bias':0.0,'cap':1600.0,'base_cap':420.0,'base_low_cap_raw':220.0,'base_sparse_cap_raw':220.0,'halluc_aux':650.0,'halluc_low':260.0,'halluc_sparse':220.0,'halluc_ratio':2.4,'halluc_sparse_ratio':2.8,'res_low':760.0,'res_sparse':260.0,'res_aux':1150.0,'res_ratio':1.45,'res_sparse_ratio':3.2,'res_mul':0.92,'corr_sparse':230.0,'corr_hi_low':1200.0,'corr_dense':700.0,'corr_hi_ratio':1.7,'corr_hi_mul':1.08,'lift_sparse':220.0,'lift_low':450.0,'lift_dense':700.0,'lift_ratio':2.0,'lift_mul':1.26,'lift_cap':1300.0,'extreme_dense':1200.0,'extreme_low':900.0,'extreme_mul':1.00,'extreme_cap':1600.0,'hi_raw':1100.0,'lo_aux':820.0,'lo_mul':1.08,'spike_raw':1300.0,'spike_aux':1050.0,'spike_cap':950.0,'hard_ratio':2.9,'hard_low':320.0},
  {'name':'strict_cap1800','scale':0.98,'gamma':1.0,'bias':0.0,'cap':1800.0,'base_cap':440.0,'base_low_cap_raw':230.0,'base_sparse_cap_raw':230.0,'halluc_aux':680.0,'halluc_low':280.0,'halluc_sparse':230.0,'halluc_ratio':2.3,'halluc_sparse_ratio':2.7,'res_low':720.0,'res_sparse':240.0,'res_aux':1080.0,'res_ratio':1.50,'res_sparse_ratio':3.1,'res_mul':0.95,'corr_sparse':230.0,'corr_hi_low':1300.0,'corr_dense':760.0,'corr_hi_ratio':1.6,'corr_hi_mul':1.06,'lift_sparse':225.0,'lift_low':430.0,'lift_dense':720.0,'lift_ratio':2.0,'lift_mul':1.28,'lift_cap':1350.0,'extreme_dense':1200.0,'extreme_low':950.0,'extreme_mul':1.00,'extreme_cap':1800.0,'hi_raw':1200.0,'lo_aux':850.0,'lo_mul':1.10,'spike_raw':1450.0,'spike_aux':1100.0,'spike_cap':1100.0,'hard_ratio':2.8,'hard_low':340.0},
  {'name':'mild_rescue','scale':1.00,'gamma':1.0,'bias':0.0,'cap':1800.0,'base_cap':460.0,'base_low_cap_raw':240.0,'base_sparse_cap_raw':240.0,'halluc_aux':700.0,'halluc_low':300.0,'halluc_sparse':240.0,'halluc_ratio':2.2,'halluc_sparse_ratio':2.6,'res_low':650.0,'res_sparse':220.0,'res_aux':980.0,'res_ratio':1.55,'res_sparse_ratio':3.0,'res_mul':0.98,'corr_sparse':240.0,'corr_hi_low':1300.0,'corr_dense':760.0,'corr_hi_ratio':1.55,'corr_hi_mul':1.05,'lift_sparse':230.0,'lift_low':420.0,'lift_dense':700.0,'lift_ratio':1.9,'lift_mul':1.30,'lift_cap':1400.0,'extreme_dense':1200.0,'extreme_low':950.0,'extreme_mul':1.00,'extreme_cap':1800.0,'hi_raw':1250.0,'lo_aux':900.0,'lo_mul':1.12,'spike_raw':1500.0,'spike_aux':1200.0,'spike_cap':1200.0,'hard_ratio':2.7,'hard_low':360.0},
 )
 for c in candidates:
  adj=np.asarray([guarded_calibrate(s,l,d,c) for s,l,d in zip(sparse,low,dense)],np.float32)
  sc_key,mt=score(a,adj); sc=float(sc_key[0])
  low_over=float(np.maximum(adj[a<160]-a[a<160],0).mean()) if (a<160).any() else 0.0
  mid_over=float(np.maximum(adj[a<420]-a[a<420],0).mean()) if (a<420).any() else 0.0
  high_over=float(np.maximum(adj[a<900]-a[a<900],0).mean()) if (a<900).any() else 0.0
  dense_under=float(np.maximum(a[(a>900)&(dense>700)]-adj[(a>900)&(dense>700)],0).mean()) if ((a>900)&(dense>700)).any() else 0.0
  spike_bad=float(np.maximum(adj[a<500]-700.0,0).mean()) if (a<500).any() else 0.0
  if low_over>18.0: sc+=4.0*(low_over-18.0)
  if mid_over>40.0: sc+=2.4*(mid_over-40.0)
  if high_over>65.0: sc+=1.4*(high_over-65.0)
  if spike_bad>0.0: sc+=3.0*spike_bad
  if dense_under>360.0: sc+=0.18*(dense_under-360.0)
  r={'strategy':f'guarded_main_{mode}','blend_alpha':0.0,'candidate':c['name'],'low_over':low_over,'mid_over':mid_over,'high_over':high_over,'dense_under':dense_under,'spike_bad':spike_bad}; r.update(c); r.update(mt); r['val_score']=sc; rows.append(r)
  if best is None or sc<best[0]: best=(sc,{'strategy':f'guarded_main_{mode}','blend_alpha':0.0,'calibrator':dict(c)})
 rows=sorted(rows,key=lambda r:(r['val_score'],r['val_p90ae'],r['val_medae'],abs(r['val_bias']))); return best[1],pd.DataFrame(rows).reset_index(drop=True)
def apply_row(r,st,modes):
 mode=modes[0]; sparse=max(0.0,float(r[f'sparse_{mode}'])); low=max(0.0,float(r[f'low_{mode}'])); dense=max(0.0,float(r[f'dense_{mode}'])); c=st['calibrator']
 c={**{'scale':1.0,'gamma':1.0,'bias':0.0,'cap':1800.0,'base_cap':450.0,'base_low_cap_raw':240.0,'base_sparse_cap_raw':240.0,'halluc_aux':700.0,'halluc_low':300.0,'halluc_sparse':240.0,'halluc_ratio':2.3,'halluc_sparse_ratio':2.7,'res_low':1e9,'res_sparse':1e9,'res_aux':1e9,'res_ratio':1.0,'res_sparse_ratio':1.0,'res_mul':1.0,'corr_sparse':0.0,'corr_hi_low':1e9,'corr_dense':1e9,'corr_hi_ratio':1e9,'corr_hi_mul':1.0,'lift_sparse':0.0,'lift_low':0.0,'lift_dense':1e9,'lift_ratio':1e9,'lift_mul':1.0,'lift_cap':1800.0,'extreme_dense':1e9,'extreme_low':1e9,'extreme_mul':1.0,'extreme_cap':1800.0,'hi_raw':1e9,'lo_aux':0.0,'lo_mul':1.0,'spike_raw':1e9,'spike_aux':1e9,'spike_cap':1800.0,'hard_ratio':1e9,'hard_low':0.0},**c}
 return guarded_calibrate(sparse,low,dense,c)


In [ ]:
seed(cfg.seed)
tr_all=samples('A','train'); te=samples('A','test'); tr,va=split(tr_all,cfg.vf,cfg.seed)
mods={}; h=[]; vf=[]
for i,mode in enumerate(cfg.modes):
 tl_sparse=DataLoader(FullDS(filter_split(tr,'sparse'),mode,cfg.gz,cfg.ds,True),batch_size=cfg.bs,shuffle=True,num_workers=0)
 vl_sparse=DataLoader(FullDS(filter_split(va,'sparse'),mode,cfg.gz,cfg.ds,False),batch_size=cfg.bs,shuffle=False,num_workers=0)
 tl_low=DataLoader(FullDS(tr,mode,cfg.gz,cfg.ds,True),batch_size=cfg.bs,shuffle=True,num_workers=0)
 vl_low=DataLoader(FullDS(va,mode,cfg.gz,cfg.ds,False),batch_size=cfg.bs,shuffle=False,num_workers=0)
 tl_dense=DataLoader(FullDS(filter_split(tr,'dense'),mode,cfg.gz,cfg.ds,True),batch_size=cfg.bs,shuffle=True,num_workers=0)
 vl_dense=DataLoader(FullDS(filter_split(va,'dense'),mode,cfg.gz,cfg.ds,False),batch_size=cfg.bs,shuffle=False,num_workers=0)
 sparse=CountNet('log').to(DEV); low=CountNet('log').to(DEV); dense=CountNet('sqrt').to(DEV)
 opt_s=torch.optim.AdamW(sparse.parameters(),lr=cfg.lr,weight_decay=cfg.wd); opt_l=torch.optim.AdamW(low.parameters(),lr=cfg.lr,weight_decay=cfg.wd); opt_d=torch.optim.AdamW(dense.parameters(),lr=cfg.lr,weight_decay=cfg.wd)
 sch_s=torch.optim.lr_scheduler.ReduceLROnPlateau(opt_s,mode='min',factor=.5,patience=3); sch_l=torch.optim.lr_scheduler.ReduceLROnPlateau(opt_l,mode='min',factor=.5,patience=3); sch_d=torch.optim.lr_scheduler.ReduceLROnPlateau(opt_d,mode='min',factor=.5,patience=3)
 best_s=deepcopy(sparse.state_dict()); best_l=deepcopy(low.state_dict()); best_d=deepcopy(dense.state_dict()); best_sv=float('inf'); best_lv=float('inf'); best_dv=float('inf'); rows=[]
 for e in range(1,cfg.ep+1):
  tm_s=ep_count(sparse,tl_sparse,opt_s,'log'); vm_s=ep_count(sparse,vl_sparse,None,'log'); sch_s.step(vm_s['mae'])
  tm_l=ep_count(low,tl_low,opt_l,'log'); vm_l=ep_count(low,vl_low,None,'log'); sch_l.step(vm_l['mae'])
  tm_d=ep_count(dense,tl_dense,opt_d,'sqrt'); vm_d=ep_count(dense,vl_dense,None,'sqrt'); sch_d.step(vm_d['mae'])
  rows.append({'mode':mode,'stage':'sparse','epoch':e,'lr':opt_s.param_groups[0]['lr'],'train_mae':tm_s['mae'],'val_mae':vm_s['mae'],'val_rmse':vm_s['rmse']})
  rows.append({'mode':mode,'stage':'main','epoch':e,'lr':opt_l.param_groups[0]['lr'],'train_mae':tm_l['mae'],'val_mae':vm_l['mae'],'val_rmse':vm_l['rmse']})
  rows.append({'mode':mode,'stage':'dense','epoch':e,'lr':opt_d.param_groups[0]['lr'],'train_mae':tm_d['mae'],'val_mae':vm_d['mae'],'val_rmse':vm_d['rmse']})
  if vm_s['mae']<best_sv: best_sv=vm_s['mae']; best_s=deepcopy(sparse.state_dict())
  if vm_l['mae']<best_lv: best_lv=vm_l['mae']; best_l=deepcopy(low.state_dict())
  if vm_d['mae']<best_dv: best_dv=vm_d['mae']; best_d=deepcopy(dense.state_dict())
 sparse.load_state_dict(best_s); low.load_state_dict(best_l); dense.load_state_dict(best_d); mods[mode]={'sparse':sparse,'low':low,'dense':dense}; h.append(pd.DataFrame(rows)); vals=[]
 for s in va:
  zs=infer_count(sparse,s['image_path'],mode,'log'); zl=infer_count(low,s['image_path'],mode,'log'); zd=infer_count(dense,s['image_path'],mode,'sqrt')
  vals.append({'image_id':s['image_id'],'actual_count':cnt(s),f'sparse_{mode}':zs['raw'],f'low_{mode}':zl['raw'],f'dense_{mode}':zd['raw'],f'sparse_den_{mode}':zs['density'],f'low_den_{mode}':zl['density'],f'dense_den_{mode}':zd['density'],f'prob_{mode}':0.0})
 vf.append(pd.DataFrame(vals))
hist=pd.concat(h,ignore_index=True); v=merge(vf); st,stab=choose(v,cfg.modes); v['raw_predicted_count']=v.apply(lambda r: apply_row(r,{'strategy':st['strategy'],'blend_alpha':st['blend_alpha'],'calibrator':{'scale':1.0,'gamma':1.0,'bias':0.0}},cfg.modes),axis=1); v['predicted_count']=v.apply(lambda r: apply_row(r,st,cfg.modes),axis=1); v['absolute_error']=(v.predicted_count-v.actual_count).abs(); print('train',len(tr),'val',len(va),'test',len(te)); print('strategy',st); display(hist.groupby(['mode','stage']).tail(3)); display(stab.head(10)); display(v.head())
tf=[]
for mode in cfg.modes:
 rows=[]
 for s in te:
  zs=infer_count(mods[mode]['sparse'],s['image_path'],mode,'log'); zl=infer_count(mods[mode]['low'],s['image_path'],mode,'log'); zd=infer_count(mods[mode]['dense'],s['image_path'],mode,'sqrt')
  rows.append({'image_id':s['image_id'],'actual_count':cnt(s),f'sparse_{mode}':zs['raw'],f'low_{mode}':zl['raw'],f'dense_{mode}':zd['raw'],f'sparse_den_{mode}':zs['density'],f'low_den_{mode}':zl['density'],f'dense_den_{mode}':zd['density'],f'prob_{mode}':0.0})
 tf.append(pd.DataFrame(rows))
p=merge(tf); p['strategy_name']=st['strategy']; p['blend_alpha']=st['blend_alpha']; p['predicted_count']=p.apply(lambda r: apply_row(r,st,cfg.modes),axis=1)
p['raw_predicted_count']=p.apply(lambda r: apply_row(r,{'strategy':st['strategy'],'blend_alpha':st['blend_alpha'],'calibrator':{'scale':1.0,'gamma':1.0,'bias':0.0}},cfg.modes),axis=1)
p['density_count_pred']=p[f'sparse_{cfg.modes[0]}']; p['aux_density_count_pred']=p[f'dense_{cfg.modes[0]}']; p['signed_error']=p.predicted_count-p.actual_count; p['absolute_error']=p.signed_error.abs(); p.insert(0,'part','A'); p.insert(1,'split','test'); p['image_path']=p.image_id.map(lambda x:f'DATASET/part_A/test_data/images/{x}.jpg'); p=p[['part','split','image_id','image_path','actual_count','density_count_pred','aux_density_count_pred','strategy_name','blend_alpha','raw_predicted_count','predicted_count','signed_error','absolute_error']]; p.to_csv(OUT,index=False); print('saved',OUT); display(p.head())
